In [1]:
!pip install networkx
!pip install stix2

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import json

with open("../data/stix21/enterprise-attack.json") as file:
    data = json.load(file)

In [3]:
nodes = []
relationships = []

for obj in data["objects"]:
    if obj["type"] == "relationship":
        relationships.append(obj)
    else:
        nodes.append(obj)

print("Number of nodes:", len(nodes))
print("Number of relationships:", len(relationships))

Number of nodes: 4724
Number of relationships: 20048


In [4]:
import networkx as nx

G = nx.DiGraph()

for node in nodes:
    node_id = node["id"]
    node_type = node["type"]
    G.add_node(node_id, type=node_type)

for rel in relationships:
    src = rel["source_ref"]
    tgt = rel["target_ref"]
    rel_type = rel["relationship_type"]

    G.add_edge(src, tgt, relationship=rel_type)

print("Graph built.")
print("Total graph nodes:", G.number_of_nodes())
print("Total graph edges:", G.number_of_edges())

Graph built.
Total graph nodes: 4724
Total graph edges: 20048


In [5]:
degree_dict = dict(G.degree())
sorted_nodes = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)
print(sorted_nodes[:10])

[('attack-pattern--e6919abc-99f9-4c6c-95a5-14761e7b2add', 501), ('attack-pattern--df8b2a25-8bdf-4856-953c-a04372b1c161', 410), ('attack-pattern--354a7f88-63fb-41b5-a801-ce3b377b36f1', 404), ('attack-pattern--d1fcf083-a721-4223-aedf-bf8960798d62', 375), ('attack-pattern--7bc57495-ea59-4380-be31-a64af124ef18', 356), ('attack-pattern--3ccef7ae-cb5e-48f6-8302-897105fbf55c', 326), ('attack-pattern--8f4a33ec-8b1f-4b80-a2f6-642b2e479580', 305), ('attack-pattern--d63a3fb8-9452-4e9d-a60a-54be68d5998c', 299), ('attack-pattern--707399d6-ab3e-4963-9315-d9d3818cd6a0', 280), ('attack-pattern--9efb1ea7-c37b-4595-9640-b7680cd84279', 255)]


In [6]:
top_ids = [sorted_nodes[i][0] for i in range(10)]

def data_node(node_id):
    for obj in data["objects"]:
        if obj["id"] == node_id:
            print(obj.get("name"))
            print(obj.get("description"))
            break

for nid in top_ids:
    data_node(nid)

Ingress Tool Transfer
Adversaries may transfer tools or other files from an external system into a compromised environment. Tools or files may be copied from an external adversary-controlled system to the victim network through the command and control channel or through alternate protocols such as [ftp](https://attack.mitre.org/software/S0095). Once present, adversaries may also transfer/spread tools between victim devices within a compromised environment (i.e. [Lateral Tool Transfer](https://attack.mitre.org/techniques/T1570)). 

On Windows, adversaries may use various utilities to download tools, such as `copy`, `finger`, [certutil](https://attack.mitre.org/software/S0160), and [PowerShell](https://attack.mitre.org/techniques/T1059/001) commands such as <code>IEX(New-Object Net.WebClient).downloadString()</code> and <code>Invoke-WebRequest</code>. On Linux and macOS systems, a variety of utilities also exist, such as `curl`, `scp`, `sftp`, `tftp`, `rsync`, `finger`, and `wget`.(Citat

In [7]:
degree = nx.degree_centrality(G)
sorted_degree = sorted(degree.items(), key=lambda x: x[1], reverse=True)

print("Top 10 by Degree:")
for node_id, score in sorted_degree[:10]:
    print(node_id, score)

Top 10 by Degree:
attack-pattern--e6919abc-99f9-4c6c-95a5-14761e7b2add 0.10607664619944951
attack-pattern--df8b2a25-8bdf-4856-953c-a04372b1c161 0.08680923142070718
attack-pattern--354a7f88-63fb-41b5-a801-ce3b377b36f1 0.08553885242430659
attack-pattern--d1fcf083-a721-4223-aedf-bf8960798d62 0.07939868727503706
attack-pattern--7bc57495-ea59-4380-be31-a64af124ef18 0.07537582045310184
attack-pattern--3ccef7ae-cb5e-48f6-8302-897105fbf55c 0.06902392547109888
attack-pattern--8f4a33ec-8b1f-4b80-a2f6-642b2e479580 0.0645775989836968
attack-pattern--d63a3fb8-9452-4e9d-a60a-54be68d5998c 0.06330721998729621
attack-pattern--707399d6-ab3e-4963-9315-d9d3818cd6a0 0.059284353165361
attack-pattern--9efb1ea7-c37b-4595-9640-b7680cd84279 0.0539911073470252


In [8]:
betweenness = nx.betweenness_centrality(G)
sorted_bet = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)

print("Top 10 by Betweenness:")
for node_id, score in sorted_bet[:10]:
    print(node_id, score)

Top 10 by Betweenness:
malware--a7881f21-e978-4fe4-af56-92c9416a2616 9.50453655502982e-05
tool--3433a9e8-1c47-4320-b9bf-ed449061d1c3 5.678011160118914e-05
tool--afc079f3-c0ea-4096-b75d-3f05338b7f60 5.3000054779042726e-05
intrusion-set--381fcf73-60f6-4ab2-9991-6af3cbc35192 3.547513468818306e-05
intrusion-set--899ce53f-13a0-479b-a0e4-67d46e241542 3.263944059561279e-05
malware--64fa0de0-6240-41f4-8638-f4ca7ed528fd 2.757023295220622e-05
intrusion-set--4ca1929c-7d64-4aab-b849-badbfc0c760d 2.5773466297157305e-05
tool--03342581-f790-4f03-ba41-e82e67392e23 2.254129469188061e-05
attack-pattern--df8b2a25-8bdf-4856-953c-a04372b1c161 2.0005399241240178e-05
intrusion-set--18854f55-ac7c-4634-bd9a-352dd07613b7 1.4901169608956883e-05


In [9]:
pagerank = nx.pagerank(G)
sorted_pr = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)

print("Top 10 by PageRank:")
for node_id, score in sorted_pr[:10]:
    print(node_id, score)

Top 10 by PageRank:
attack-pattern--7385dfaf-6886-4229-9ecd-6fd678040830 0.009250532816894712
attack-pattern--b3d682b6-98f2-4fb0-aa3b-b4df007ca70a 0.008928012955156423
attack-pattern--1ecb2399-e8ba-4f6b-8ba7-5c27d49405cf 0.006258568532663142
attack-pattern--b6301b64-ef57-4cce-bb0b-77026f14a8db 0.005855723222869892
attack-pattern--799ace7f-e227-4411-baa0-8868704f2a69 0.005560128014989519
attack-pattern--42e8de7b-37b2-4258-905a-6897815e58e0 0.005153269858099451
attack-pattern--355be19c-ffc9-46d5-8d50-d6a036c675b6 0.005093711858945087
attack-pattern--457c7820-d331-465a-915e-42f85500ccc4 0.004970237138265317
attack-pattern--e6919abc-99f9-4c6c-95a5-14761e7b2add 0.004772549635625029
attack-pattern--22905430-4901-4c2a-84f6-98243cb173f8 0.00465493963522257


In [22]:
combined_score = {}

for node in G.nodes():
    d = degree.get(node, 0)
    b = betweenness.get(node, 0)
    p = pagerank.get(node, 0)

    score = 0.4*d + 0.3*b + 0.3*p
    combined_score[node] = score
sorted_combined = sorted(combined_score.items(), key=lambda x: x[1], reverse=True)

print("Top 10 by Combined Risk Score:")
for node_id, score in sorted_combined[:10]:
    print(node_id, score)

Top 10 by Combined Risk Score:
attack-pattern--e6919abc-99f9-4c6c-95a5-14761e7b2add 0.04386242337046732
attack-pattern--df8b2a25-8bdf-4856-953c-a04372b1c161 0.035867061193813485
attack-pattern--354a7f88-63fb-41b5-a801-ce3b377b36f1 0.035264500162805995
attack-pattern--d1fcf083-a721-4223-aedf-bf8960798d62 0.03279602838468275
attack-pattern--7bc57495-ea59-4380-be31-a64af124ef18 0.03111014388869845
attack-pattern--3ccef7ae-cb5e-48f6-8302-897105fbf55c 0.02847915053649113
attack-pattern--8f4a33ec-8b1f-4b80-a2f6-642b2e479580 0.026589535137300845
attack-pattern--d63a3fb8-9452-4e9d-a60a-54be68d5998c 0.02608915863071377
attack-pattern--707399d6-ab3e-4963-9315-d9d3818cd6a0 0.024666357065387938
attack-pattern--9efb1ea7-c37b-4595-9640-b7680cd84279 0.022280733014365933


In [27]:
top_ids = [node_id for node_id, score in sorted_combined[:10]]

def data_node(node_id):
    for obj in data["objects"]:
        if obj["id"] == node_id:
            print(obj.get("name"))
            print(obj.get("description"))
            break

for nid in top_ids:
    data_node(nid)

Ingress Tool Transfer
Adversaries may transfer tools or other files from an external system into a compromised environment. Tools or files may be copied from an external adversary-controlled system to the victim network through the command and control channel or through alternate protocols such as [ftp](https://attack.mitre.org/software/S0095). Once present, adversaries may also transfer/spread tools between victim devices within a compromised environment (i.e. [Lateral Tool Transfer](https://attack.mitre.org/techniques/T1570)). 

On Windows, adversaries may use various utilities to download tools, such as `copy`, `finger`, [certutil](https://attack.mitre.org/software/S0160), and [PowerShell](https://attack.mitre.org/techniques/T1059/001) commands such as <code>IEX(New-Object Net.WebClient).downloadString()</code> and <code>Invoke-WebRequest</code>. On Linux and macOS systems, a variety of utilities also exist, such as `curl`, `scp`, `sftp`, `tftp`, `rsync`, `finger`, and `wget`.(Citat